In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = MNIST(root="./data", train=True, download=True, transform=transform)
test_set = MNIST(root="./data", train=False, download=True, transform=transform)

In [3]:
trainloader = DataLoader(train_set, batch_size=64, shuffle=True)
testloader = DataLoader(test_set, batch_size=64)

# Building the RNN

In [4]:
class RNN(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=1):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])

        return out

In [5]:
device = torch.device("cpu")

In [6]:
model = RNN().to(device)

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
state_dict = torch.load("RNN_parameters.pth")
model.load_state_dict(state_dict)

<All keys matched successfully>

In [ ]:
# Once the model's best parameters are saved then no need to run this cell again 

epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for Xb, yb in trainloader:
        Xb = Xb.view(Xb.size(0), 28, 28).to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        outputs = model(Xb)

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_loss/len(trainloader)}")

epoch=1/10 & loss=0.7917469760248148
epoch=2/10 & loss=0.3507136565917082
epoch=3/10 & loss=0.2546861737148403
epoch=4/10 & loss=0.21328624080159644
epoch=5/10 & loss=0.17482239565949068
epoch=6/10 & loss=0.1628694124202103
epoch=7/10 & loss=0.14382597187887441
epoch=8/10 & loss=0.1350364777624909
epoch=9/10 & loss=0.12296581736354749
epoch=10/10 & loss=0.11981033196033382


In [11]:
torch.save(model.state_dict(), "RNN_parameters.pth")

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    
    for Xb, yb in testloader:
        Xb = Xb.view(Xb.size(0), 28, 28).to(device)
        yb = yb.to(device)

        outputs = model(Xb)

        _, predicted = torch.max(outputs, 1)

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    print(f"Accuracy Score: {correct_vals / total_vals * 100}%")
    print(f"Confusion Matrix: \n", confusion_matrix(all_labels, all_preds))

Accuracy Score: 96.33%
Confusion Matrix: 
 [[ 963    0    2    0    0    2    7    1    4    1]
 [   0 1114    2    3    1    3    2    1    8    1]
 [   2    5  995    4    0    0    5   12    9    0]
 [   0    0    9  978    0    3    0    6    6    8]
 [   0    1    6    1  913    0    8    0    6   47]
 [   3    1    0   33    0  812    9    1   16   17]
 [   3    2    0    1    3    1  932    0   16    0]
 [   3    1    6    2    0    2    0  997    3   14]
 [   1    5    3    2    1    2    1    4  951    4]
 [   2    1    0    8    3    1    0    7    9  978]]
